# 🎯 Regresión logística: predecir "sí" o "no" (versión sencilla)
**Curso de IA · Bloque de Machine Learning · Sesión 4 (versión básica)**

---

## Un cambio importante: de números a categorías

Hasta ahora **predecíamos un número** (el precio de un piso, el coste de un seguro). Hoy predecimos **una respuesta de tipo "sí / no"**:

| Antes (regresión) | Hoy (clasificación) |
|---|---|
| ¿Cuánto costará este piso? | ¿Este cliente **abandonará** la compañía? |
| Resultado: un número | Resultado: una **categoría** (sí / no) |

## El problema de hoy

Una compañía de telefonía quiere saber **qué clientes están a punto de darse de baja**, para ofrecerles una promoción **antes** de que se vayan.

## La idea de la regresión logística

Aunque se llama "regresión", sirve para **clasificar**. Funciona así:

1. Combina las características del cliente en una **puntuación**.
2. Convierte esa puntuación en una **probabilidad** (un número entre 0 y 1).
3. Si la probabilidad es **mayor que 0,5** → predice **"sí abandona"**. Si no → **"no abandona"**.

> 💬 **En una frase:** la regresión logística **calcula la probabilidad** de que ocurra algo, y luego decidimos a partir de qué probabilidad damos la alarma.

## Los pasos de hoy

| Apartado | Paso | En una frase |
|---|------|--------------|
| 1-2 | Herramientas y datos | Librerías y CSV |
| 3 | Cargar y mirar | ¿Cuántos abandonan? |
| 4 | Explorar | ¿Qué tipo de cliente se va más? |
| 5 | Texto a números | `contrato` → 0 / 1 |
| 6 | Separar entrenamiento y prueba | Guardar clientes "para el examen" |
| 7 | **La trampa del accuracy** | Un modelo que no hace nada acierta casi siempre |
| 8 | Entrenar la regresión logística | `fit` y `predict` |
| 9 | Medir de verdad | Matriz de confusión, precision y recall |
| 10 | Cambiar el umbral | ¿Y si avisamos con menos seguridad? |
| 11 | Predecir clientes nuevos | Usar el modelo |
| 12 | Resumen | La receta |

## Cómo usar este cuaderno

Ejecuta cada celda con **Shift + Enter**, **en orden**. Si algo se estropea: *Entorno de ejecución → Reiniciar y ejecutar todo*. Antes de ejecutar, **lee y piensa qué va a salir**.

> 👩‍🏫 **Nota para el profesor**
>
> Versión **simplificada** del cuaderno de regresión logística. Se han quitado: nulos y texto sucio, el `Pipeline`/`ColumnTransformer`, la sigmoide dibujada, los *odds ratio*, la curva ROC y el AUC, `class_weight`, la comparación con otros modelos y la validación cruzada. Solo hay **una** variable de texto (`contrato`), y el centro de la sesión es la **matriz de confusión** y el **umbral**.
>
> **Duración orientativa: unas 3 horas** con los ejercicios.

## 1. Preparar las herramientas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Todo listo ✅")

## 2. Crear el fichero de datos

Esta celda fabrica `clientes.csv` con **600 clientes**.

| Columna | Significado |
|---|---|
| `antiguedad` | Meses como cliente |
| `cuota` | Cuota mensual (€) |
| `llamadas_soporte` | Llamadas al servicio de atención en los últimos 3 meses |
| `contrato` | `mensual` (sin permanencia) o `anual` ← **texto** |
| `abandona` | `sí` si se dio de baja, `no` si sigue ← **lo que queremos predecir** |

> 💡 No hace falta entender el código de esta celda.

In [ ]:
rng = np.random.default_rng(21)
n = 600

filename = "./csv/clientes.csv"

antiguedad = rng.integers(1, 73, n)
cuota = rng.normal(55, 15, n).clip(20, 110).round(1)
llamadas = rng.poisson(1.5, n)
contrato = rng.choice(["mensual", "anual"], n, p=[0.55, 0.45])

z = (-2.0 - 0.04 * antiguedad + 0.03 * (cuota - 55) + 0.6 * llamadas + 1.6 * (contrato == "mensual"))
probabilidad = 1 / (1 + np.exp(-z))
abandona = np.where(rng.random(n) < probabilidad, "sí", "no")

pd.DataFrame({
    "antiguedad": antiguedad, "cuota": cuota, "llamadas_soporte": llamadas,
    "contrato": contrato, "abandona": abandona,
}).to_csv(filename, index=False, encoding="utf-8")

print(f"✅ Fichero '{filename}' creado")

## 3. Cargar y mirar

Lo primero en clasificación: **¿cuántos hay de cada clase?**

In [ ]:
df = pd.read_csv(filename)
print("Filas y columnas:", df.shape)
df.head(8)

In [ ]:
cuantos = df["abandona"].value_counts()
display(cuantos)
porcentaje = (df["abandona"].value_counts(normalize=True) * 100).round(1)
display(porcentaje)

pd.DataFrame({"clientes": cuantos, "%": porcentaje})

👀 **Se quedan más de los que se van** (las clases están **desbalanceadas**: unas son más frecuentes que otras). Esto es lo normal en la vida real, y tiene consecuencias importantes que veremos en el apartado 7.

## 4. Explorar: ¿qué clientes se van más?

El **porcentaje de abandono** según el tipo de contrato. (Truco: convertimos "sí" en 1 y "no" en 0; la media de unos y ceros es el porcentaje de unos.)

In [ ]:
df["abandona_num"] = (df["abandona"] == "sí").astype(int)

tasa = (df.groupby("contrato")["abandona_num"].mean() * 100).round(1)
print(tasa)

tasa.plot(kind="bar", color=["steelblue", "tomato"], figsize=(5, 3.5))
plt.title("% de clientes que abandonan según el contrato")
plt.ylabel("% que abandona")
plt.xticks(rotation=0)
plt.show()

In [ ]:
se_queda = df[df["abandona"] == "no"]["antiguedad"]
se_va = df[df["abandona"] == "sí"]["antiguedad"]
display(se_va)


plt.boxplot([se_queda, se_va])
plt.xticks([1, 2], ["Se queda", "Abandona"])
plt.ylabel("Antigüedad (meses)")
plt.title("Los que abandonan suelen tener menos antigüedad")
plt.show()

👀 **Pistas:**
- Los clientes con contrato **mensual** (sin permanencia) abandonan **mucho más**.
- Los que **abandonan** suelen tener **menos antigüedad**.

El modelo debería descubrir esto por sí solo.

## 5. Texto a números

`contrato` es texto. Lo traducimos: **1 si es mensual**, **0 si es anual**. Y la variable objetivo `abandona`: **1 = sí abandona** (lo que queremos detectar) y **0 = no**.

In [ ]:
df["contrato_mensual"] = (df["contrato"] == "mensual").astype(int)
df["abandona"] = df["abandona_num"]

df = df.drop(columns=["contrato", "abandona_num"])
df.head()

## 6. Separar entrenamiento y prueba

Como siempre: una parte para **aprender** y otra para **el examen**. Con clases desbalanceadas añadimos `stratify=y`, que hace que **la proporción de abandonos sea igual en las dos partes** (si no, por azar el examen podría tener muy pocos casos).

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="abandona")
y = df["abandona"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("% de abandonos en entrenamiento:", round(y_train.mean() * 100, 1))
print("% de abandonos en prueba       :", round(y_test.mean() * 100, 1))

## 7. La trampa del accuracy

La medida más intuitiva es el **accuracy** (exactitud): **el porcentaje de aciertos**.

Probemos un modelo **absurdo** que **nunca avisa de ningún abandono**: siempre dice "no abandona".

In [ ]:
from sklearn.metrics import accuracy_score

prediccion_tonta = np.zeros(len(y_test))      # siempre predice 0 = "no abandona"

print(f"Accuracy del modelo que no hace nada: {accuracy_score(y_test, prediccion_tonta):.2f}")
print(f"Abandonos que detecta: {int((prediccion_tonta[y_test == 1] == 1).sum())} de {int(y_test.sum())}")

🚨 **¡Un modelo que no hace NADA acierta alrededor del 75 % de las veces!** Solo porque la mayoría de clientes se queda. Pero **no detecta ni un solo abandono**, que es justo lo que queríamos.

> 🧠 **Lección clave:** con clases desbalanceadas, **el accuracy engaña**. Hay que mirar otras medidas (*precision* y *recall*) que veremos enseguida.

## 8. Entrenar la regresión logística

Con scikit-learn son **las mismas tres líneas de siempre**: crear, entrenar (`fit`), predecir (`predict`).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
filas = []
for profundidad in range(1, 11):
    a = DecisionTreeClassifier(max_depth=profundidad, random_state=42).fit(X_train, y_train)
    filas.append({
        "profundidad": profundidad,
        "hojas": a.get_n_leaves(),
        "accuracy entrenamiento": round(accuracy_score(y_train, a.predict(X_train)), 3),
        "accuracy examen": round(accuracy_score(y_test, a.predict(X_test)), 3),
    })

tabla = pd.DataFrame(filas).set_index("profundidad")
depth_op = tabla["accuracy examen"].idxmax()
display(tabla)
print(f"La profundidad optima optenida es: {depth_op}")

In [ ]:
arbol = DecisionTreeClassifier(max_depth=depth_op, random_state=42)
arbol.fit(X_train, y_train)
predicionesArbolin = arbol.predict(X_test)
print("✅ Modelo entrenado")
print(f"Accuracy en prueba: {accuracy_score(y_test, predicionesArbolin):.2f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

arbol = DecisionTreeClassifier(max_depth=depth_op, random_state=42)
arbol.fit(X_train, y_train)

bosque = RandomForestClassifier(n_estimators=200, max_depth=depth_op, random_state=42)
bosque.fit(X_train, y_train)

predicciones = modelo.predict(X_test)
prediccionesArbol = arbol.predict(X_test)
prediccionesBosque = bosque.predict(X_test)

fila = []
for i, model in enumerate([modelo, arbol, bosque]):

    fila.append({
        "modelo": map({0: "Logistic", 1:"Arbol", 2: "Bosque"}),
        "accuracy entrenamiento": round(accuracy_score(y_train, model.predict(X_train)), 3),
        "accuracy examen": round(accuracy_score(y_test, model.predict(X_test)), 3),
    })
print("✅ Modelo entrenado")
print(f"Accuracy en prueba: {accuracy_score(y_test, predicciones):.2f}")
tabla = pd.DataFrame(filas)
display(tabla)

### Dos formas de predecir

- `predict()` → la **clase** directamente (0 o 1).
- `predict_proba()` → la **probabilidad** de cada clase. Es mucho más informativa: un cliente con 51 % y otro con 95 % se clasifican igual, pero **no son lo mismo**.

In [ ]:
probabilidades = modelo.predict_proba(X_test)[:, 1]      # probabilidad de abandonar

tabla = X_test.copy()
tabla["prob. de abandonar"] = (probabilidades * 100).round(0).astype(int).astype(str) + " %"
tabla["predice"] = predicciones
tabla["realidad"] = y_test.values
tabla.head(10)

👀 Fíjate: cuando `prob. de abandonar` supera el 50 %, `predice` vale 1. Compara `predice` con `realidad`: **hay aciertos y errores**.

### ¿Qué ha aprendido? Los coeficientes

El **signo** de cada coeficiente dice **hacia dónde empuja** esa variable:
- **Positivo (+)** → **aumenta** la probabilidad de abandonar.
- **Negativo (−)** → la **reduce**.

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": modelo.coef_[0].round(2),
})
coeficientes

👀 **Cómo leerlo:**
- `contrato_mensual` (**positivo, el mayor**): tener contrato mensual **dispara** el riesgo.
- `llamadas_soporte` (**positivo**): cuantas más llamadas, más riesgo.
- `antiguedad` (**negativo**): cuanta más antigüedad, **menos** riesgo.

¡Justo lo que habíamos visto en los gráficos!

## 9. Medir de verdad: la matriz de confusión

Para ver **cómo se equivoca** el modelo usamos la **matriz de confusión**: una tabla que cruza **lo que pasó de verdad** con **lo que predijo el modelo**.

|  | Predice **"no abandona"** | Predice **"abandona"** |
|---|---|---|
| **Realmente no abandona** | ✅ **Acierto** (verdadero negativo) | ❌ **Falsa alarma** (falso positivo) |
| **Realmente abandona** | ❌ **Se nos escapa** (falso negativo) | ✅ **Acierto** (verdadero positivo) |

En nuestro problema:
- **Falsa alarma:** ofrecemos una promoción a alguien que **no** se iba a ir → gastamos dinero de más.
- **Se nos escapa:** no hacemos nada y el cliente **se va** → perdemos al cliente.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, predicciones, display_labels=["No abandona", "Abandona"], cmap="Blues", colorbar=False
)
plt.title("Matriz de confusión")
plt.show()

### Dos medidas que sí sirven

| Medida | Pregunta que responde | Ejemplo |
|---|---|---|
| **Precision** | De los clientes a los que **avisamos**, ¿cuántos **realmente** se iban? | "Avisé a 10 y 8 se iban → 80 %" |
| **Recall** | De los clientes que **realmente se iban**, ¿a cuántos **detecté**? | "Se iban 20 y detecté 8 → 40 %" |

> 🎣 **Analogía de la pesca:** el **recall** es lo **grande que es la red** (cuántos peces atrapas). La **precision** es lo **selectiva** que es (cuánto de lo que atrapas es pescado de verdad y no basura).

In [ ]:
from sklearn.metrics import precision_score, recall_score

print(f"Precision: {precision_score(y_test, predicciones):.2f}   (cuando avisamos, acertamos esta proporción)")
print(f"Recall   : {recall_score(y_test, predicciones):.2f}   (detectamos esta proporción de los abandonos)")

👀 **Compara con el modelo que no hacía nada:** él detectaba el **0 %** de los abandonos. Nuestro modelo detecta **una parte**: ya **sirve para algo**. Pero **seguramente se nos escapan bastantes**. ¿Podemos hacer algo?

## 10. Cambiar el umbral

Hasta ahora avisábamos cuando la probabilidad **superaba el 50 %**. Pero ese 50 % es solo una **costumbre**. Si **perder un cliente es muy caro**, podemos **avisar con menos seguridad**, por ejemplo a partir del **30 %**.

Veamos qué pasa con distintos umbrales:

In [ ]:
filas = []
for umbral in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    avisados = (probabilidades >= umbral).astype(int)
    filas.append({
        "umbral": umbral,
        "clientes avisados": int(avisados.sum()),
        "precision": round(precision_score(y_test, avisados, zero_division=0), 2),
        "recall": round(recall_score(y_test, avisados), 2),
    })

pd.DataFrame(filas)

👀 **Es un equilibrio:**
- **Umbral bajo** (0,2-0,3): avisamos a **muchos** → el **recall sube** (detectamos más abandonos) pero la **precision baja** (más falsas alarmas).
- **Umbral alto** (0,6-0,7): avisamos a **pocos**, solo cuando estamos muy seguros → la **precision sube** pero el **recall baja** (se nos escapan más).

**No se pueden mejorar las dos a la vez.** Hay que **elegir según lo que cueste cada error**:

| Si es muy caro... | Conviene... |
|---|---|
| **Perder un cliente** (falso negativo) | Umbral **bajo** (más recall) |
| **Molestar o gastar en vano** (falsa alarma) | Umbral **alto** (más precision) |

> 🎯 El modelo es el mismo; lo que cambia es **cómo decidimos usarlo**.

## 11. Predecir clientes nuevos

Describimos clientes con **las mismas columnas** que en el entrenamiento y pedimos su **probabilidad** de abandonar:

In [ ]:
clientes_nuevos = pd.DataFrame({
    "antiguedad":       [3, 48, 12],
    "cuota":            [80.0, 40.0, 65.0],
    "llamadas_soporte": [4, 0, 2],
    "contrato_mensual": [1, 0, 1],
})

prob = modelo.predict_proba(clientes_nuevos[X.columns])[:, 1]

resultado = clientes_nuevos.copy()
resultado["prob. de abandonar"] = (prob * 100).round(0).astype(int).astype(str) + " %"
resultado["decisión (umbral 30 %)"] = np.where(prob >= 0.3, "⚠️ Ofrecer promoción", "✅ No hacer nada")
resultado

Las **probabilidades** permiten **ordenar** a los clientes por riesgo y contactar primero a los más peligrosos.

## 12. Resumen

### La receta de clasificación

| Paso | Código |
|---|---|
| 1. ¿Cuántos hay de cada clase? | `df["col"].value_counts()` |
| 2. Texto → números | `.map(...)` / `(df["col"] == "valor").astype(int)` |
| 3. Separar (¡con `stratify`!) | `train_test_split(X, y, test_size=0.2, stratify=y)` |
| 4. Entrenar | `LogisticRegression().fit(X_train, y_train)` |
| 5. Predecir clase o probabilidad | `.predict(...)` / `.predict_proba(...)` |
| 6. Medir | matriz de confusión, `precision_score`, `recall_score` |
| 7. Ajustar el umbral | `probabilidades >= 0.3` |

### Ideas clave

1. En clasificación se predicen **categorías**; la regresión logística calcula **probabilidades**.
2. Con clases desbalanceadas, **el accuracy engaña**: un modelo que no hace nada puede acertar el 70 %.
3. La **matriz de confusión** muestra los cuatro casos: aciertos, falsas alarmas y casos que se escapan.
4. **Precision** = "de los que aviso, cuántos acierto". **Recall** = "de los reales, cuántos detecto".
5. El **umbral** se elige según **lo que cueste cada tipo de error**.

### 📖 Mini glosario

| Palabra | Significado sencillo |
|---|---|
| **Clasificación** | Predecir una categoría (sí/no, tipo A/B/C) |
| **Clase positiva** | La que queremos detectar (aquí, "abandona") |
| **Probabilidad** | Un número entre 0 y 1 (0 % a 100 %) |
| **Umbral** | Probabilidad a partir de la cual damos la alarma |
| **Falso positivo** | Falsa alarma |
| **Falso negativo** | Caso real que se nos escapa |

## ✍️ Ejercicios

**Nivel 1 – Para empezar**
1. En la matriz de confusión, cuenta **a mano** cuántos clientes hay en cada casilla y comprueba que suman el total de clientes de prueba.
2. Calcula a mano la **precision** y el **recall** con esos números y compara con los que dio scikit-learn.
3. ¿Qué pasaría si el umbral fuera **0** (avisamos a todos)? ¿Y **1** (no avisamos a nadie)? ¿Cuánto valdrían el recall y la precision?

**Nivel 2 – Practicar**
4. Cambia el cliente nuevo número 3 a contrato **anual** (`contrato_mensual = 0`). ¿Cuánto baja su probabilidad de abandonar?
5. Quita la columna `cuota` y vuelve a entrenar. ¿Cambian mucho el accuracy, la precision y el recall?
6. Cambia `test_size` a 0.3. ¿Cambian las medidas? ¿Por qué?

**Nivel 3 – Reto**
7. Supón que **perder un cliente cuesta 300 €** y **contactar a uno cuesta 30 €**. Para cada umbral de la tabla, calcula el coste total: `30 × (avisados) + 300 × (abandonos no detectados)`. ¿Qué umbral sale más barato?

---
### 🎯 ¿Y ahora qué?
Seguiremos con modelos de **clasificación** más flexibles: los **árboles de decisión** y los **bosques aleatorios**.